In [12]:
import requests
import json
import os
import time
from datetime import datetime

In [13]:
# === CONFIGURATION ===

BASE_DIR = "../JCDL_Code_2022_2025/Scientific_Novelty_Detection_2022_2025"
CACHE_DIR = os.path.join(BASE_DIR, "cache")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

YEAR_START = 2022
YEAR_END = 2025

PER_PAGE = 200   # OpenAlex max allowed
MAX_RETRIES = 5

# === TASK DEFINITIONS (aligned with 2021 categories) ===

TASKS = {
    "Dia2022_2025": ["dialogue", "conversation", "chat"],
    "MT2022_2025": ["machine translation", "translation"],
    "QA2022_2025": ["question answering", "reading comprehension"],
    "SA2022_2025": ["sentiment analysis", "emotion"],
    "Sum2022_2025": ["summarization"]
}

In [14]:
def safe_request(url):
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(url, timeout=30)

            if response.status_code == 429:
                sleep_time = 2 ** attempt
                print(f"Rate limited. Sleeping {sleep_time}s")
                time.sleep(sleep_time)
                continue

            response.raise_for_status()
            return response.json()

        except Exception as e:
            sleep_time = 2 ** attempt
            print(f"Error: {e}. Retrying in {sleep_time}s")
            time.sleep(sleep_time)

    return None

In [15]:
def fetch_task_metadata(task_name, keywords):

    cache_file = os.path.join(CACHE_DIR, f"{task_name}_metadata.json")
    checkpoint_file = os.path.join(CHECKPOINT_DIR, f"{task_name}_fetch_checkpoint.json")

    # If already cached completely, load and return
    if os.path.exists(cache_file):
        print(f"Loading cached metadata for {task_name}")
        with open(cache_file, "r") as f:
            return json.load(f)

    # Load checkpoint if exists
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            checkpoint = json.load(f)
        page = checkpoint["page"]
        papers = checkpoint["papers"]
    else:
        page = 1
        papers = []

    print(f"Fetching metadata for {task_name}")

    while True:
        url = (
            f"https://api.openalex.org/works?"
            f"filter=publication_year:{YEAR_START}-{YEAR_END}"
            f"&per-page={PER_PAGE}&page={page}"
        )

        data = safe_request(url)

        if data is None:
            break

        results = data.get("results", [])
        if not results:
            break

        for paper in results:
            title = (paper.get("title") or "").lower()

            # reconstruct abstract from inverted index
            abstract_index = paper.get("abstract_inverted_index")
            abstract = ""
            if abstract_index:
                words = []
                for word, positions in abstract_index.items():
                    for pos in positions:
                        words.append((pos, word))
                words = sorted(words)
                abstract = " ".join([w[1] for w in words]).lower()

            full_text = title + " " + abstract

            if any(keyword in full_text for keyword in keywords):
                papers.append({
                    "id": paper.get("id"),
                    "title": paper.get("title"),
                    "abstract": abstract,
                    "year": paper.get("publication_year"),
                    "pdf_url": paper.get("open_access", {}).get("oa_url")
                })

        # Save checkpoint after each page
        with open(checkpoint_file, "w") as f:
            json.dump({"page": page + 1, "papers": papers}, f, indent=2)

        print(f"{task_name} — Page {page} processed. Papers collected: {len(papers)}")

        page += 1
        time.sleep(1)

    # Final save to cache
    with open(cache_file, "w") as f:
        json.dump(papers, f, indent=2)

    print(f"{task_name} metadata fetch complete. Total papers: {len(papers)}")

    return papers

In [16]:
all_metadata = {}

for task, keywords in TASKS.items():
    papers = fetch_task_metadata(task, keywords)
    all_metadata[task] = papers

print("All tasks metadata fetched.")

Loading cached metadata for Dia2022_2025
Loading cached metadata for MT2022_2025
Loading cached metadata for QA2022_2025
Loading cached metadata for SA2022_2025
Loading cached metadata for Sum2022_2025
All tasks metadata fetched.
